# Exercise 3: Inversion (18 min)

`pytem.invert` minimises

$$\phi(\mathbf{m}) = \|\mathbf{W}(\ln \mathbf{d}_{obs} - \ln \mathbf{d}_{pred}(\mathbf{m}))\|^2 + \alpha\, \mathbf{m}^T \mathbf{R}\, \mathbf{m}$$

with Gauss-Newton steps in $\mathbf{m} = \ln \rho$, lowering $\alpha$ until the data RMS reaches 1. You will make synthetic data from a known model, add noise, invert, and compare the result with the truth.

### Functions you will need
```python
pytem.fwd_analytical_central(resistivity, tx_radius, times)          # half-space, closed form
pytem.fwd_circle_central(thicknesses, resistivities, tx_radius, times,
                         transform='dlf',            # or 'euler'
                         hankel_filter='key_101',    # or 'key_201'
                         fourier_filter='key_81')    # or 'key_101'
pytem.dbdt_to_apprho(dbdt, tx_moment, times)         # tx_moment = loop area x current [A m²]
pytem.plot_model(thicknesses, resistivities, ax=ax, label='...')
```
- `resistivities` has N entries, `thicknesses` N-1 (the last layer is the half-space). A half-space is `thicknesses=[]`, `resistivities=[rho]`.
- All forward functions return dB/dt in V/m² for 1 A. The step-off response is **negative**, so plot `-d` on log axes.

```python
res = pytem.invert(d_obs,            # observed -dB/dt, POSITIVE values
                   thk_inv,          # (N-1,) inversion layer thicknesses [m]
                   m0,               # (N,) starting ln(resistivity)
                   a, times,         # loop radius [m], gate times [s]
                   noise_std=0.03,   # relative data error (float) or absolute error per gate (array)
                   max_noise_frac=0.0,
                   regularization='l2',   # or 'l1'
                   maxit=15)
res['resistivities']   # (N,) final model [Ohm.m]
res['rms_history']     # RMS after each iteration
```

In [ ]:
# pip install pytem
import time
import numpy as np
import matplotlib.pyplot as plt
import pytem

a = np.sqrt(40**2 / np.pi)
times = np.logspace(-5, -2, 31)
thk_true, rho_true = [30, 20], [100, 10, 300]     # true model

# 25 inversion layers, thickness increasing with depth, down to 150 m
n_layers = 25
thk_inv = np.diff(np.r_[0, np.geomspace(2, 150, n_layers - 1)])

rng = np.random.default_rng(1)

### Task 3.1: synthetic data
Compute the response of the true model, take `-d` so it is positive, and add 3 % Gaussian noise: multiply each gate by `1 + 0.03 * rng.standard_normal(len(times))`. Plot the clean and noisy data.

In [ ]:
d_true = -pytem.fwd_circle_central(thk_true, rho_true, a, times)
d_obs = d_true * (1 + 0.03 * rng.standard_normal(len(times)))

plt.loglog(times, d_true, 'k-', label='Clean')
plt.loglog(times, d_obs, '.', label='3 % noise')
plt.xlabel('Time [s]'); plt.ylabel('-dB/dt [V/m²]'); plt.legend()

### Task 3.2: smooth vs blocky inversion
Make a starting model of 100 Ohm.m in every layer (`m0` is ln of resistivity). Invert the noisy data with `regularization='l2'` and with `'l1'`.

Make a three-panel figure: observed and predicted data (compute the predicted data with the forward model and the final resistivities), RMS against iteration, and the two models with the true model.

Which recovers the interfaces better? Which recovers the clay resistivity better? Which parts of the model are poorly recovered, and why?

In [ ]:
m0 = np.full(n_layers, np.log(100.0))

def run(d_obs, noise, runs):
    fig, axs = plt.subplots(1, 3, figsize=(14, 4))
    axs[0].loglog(times, d_obs, 'k.', label='Observed')
    for label, kw in runs.items():
        res = pytem.invert(d_obs, thk_inv, kw.pop('m0', m0), a, times, noise_std=noise,
                           max_noise_frac=0.0, maxit=15, **kw)
        d_pred = -pytem.fwd_circle_central(thk_inv, res['resistivities'], a, times)
        axs[0].loglog(times, d_pred, label=label)
        axs[1].plot(res['rms_history'], 'o-', label=label)
        pytem.plot_model(thk_inv, res['resistivities'], ax=axs[2], label=label)
    pytem.plot_model(thk_true, rho_true, ax=axs[2], label='True', color='k', depth_pad=110)
    axs[2].set_ylim(160, 0)
    axs[0].set(xlabel='Time [s]', ylabel='-dB/dt [V/m²]'); axs[0].legend()
    axs[1].set(xlabel='Iteration', ylabel='RMS'); axs[1].legend()
    plt.tight_layout()

run(d_obs, 0.03, {'L2 (smooth)': dict(regularization='l2'),
                  'L1 (blocky)': dict(regularization='l1')})

### Task 3.3: noise
Make data with 1 % and 10 % noise and invert each (L2), telling the inversion the correct noise. How does the recovered model change? Why does the inversion not fit the 10 % data exactly?

In [ ]:
for nf in [0.01, 0.10]:
    d_n = d_true * (1 + nf * rng.standard_normal(len(times)))
    run(d_n, nf, {f'{nf:.0%} noise': dict(regularization='l2')})

### Task 3.4: a wrong noise estimate
Invert the 3 % data while telling the inversion the noise is 0.5 %, then 20 %. What happens to the final RMS and to the model? What does this mean for field data?

In [ ]:
run(d_obs, 0.005, {'assumed 0.5 %': dict(regularization='l2')})
run(d_obs, 0.20, {'assumed 20 %': dict(regularization='l2')})

### Task 3.5 (if time)
- Start from 10 Ohm.m and from 1000 Ohm.m. Does the result depend on the starting model?
- Invert the three equivalent models from Exercise 2.3. Can the inversion tell them apart?
- Move the clay layer to 100 m depth. Where does the model stop following the truth?

In [ ]:
run(d_obs, 0.03, {'start 10 Ohm.m': dict(m0=np.full(n_layers, np.log(10.0))),
                  'start 1000 Ohm.m': dict(m0=np.full(n_layers, np.log(1000.0)))})